### Script to Run Trajcetory, Adjoint, and TLM.

What to do first:
* Create a production directory (if it doesn't exist already) and place "insert_file_name_here.ipynb", "wrf_settings.py", and "make_namelist.py".
* In that directory create an "em_adj, em_tlm" directory.
* In em_adj, the following files are required:
    * GENPARM.TBL
    * LANDUSE.TBL
    * RRTMG_LW_DATA
    * RRTMG_SW_DATA
    * RRTM_DATA
    * VEGPARM.TBL
    * wrf.exe
    * plus.io_config
    * wrfbdy_d01
    * wrfinput_d01


In [1]:
import numpy as np
import netCDF4
from netCDF4 import Dataset
from wrf import getvar
import os
import subprocess
from subprocess import PIPE
import sys
import glob
import importlib
import pandas as pd
import xarray as xr

sys.path.append(os.path.abspath('..'))

import wrf_settings
import make_namelist

In [2]:
#Select experiment to load settings for

EXP_NAME = 'WRF_Florence_test'

importlib.reload(wrf_settings)

settings = wrf_settings.get_settings(EXP_NAME)
WRF_DIR           = settings['WRF_dir']
BOX_SIZE          = settings['box_size']
ADJ_JC            = settings['adj_jc']
ADJ_IC            = settings['adj_ic']

RUN_HOURS         = settings['run_hours']
START_YEAR          = settings['start_year']
START_MONTH         = settings['start_month']
START_DAY           = settings['start_day']
START_HOUR          = settings['start_hour']
END_YEAR            = settings['end_year']
END_MONTH           = settings['end_month']
END_DAY             = settings['end_day']
END_HOUR            = settings['end_hour']
E_WE               = settings['e_we']
E_SN               = settings['e_sn']
DX                 = settings['dx']
DY                 = settings['dy']
TIME_STEP          = settings['time_step']
INTERVAL_SECONDS   = settings['interval_seconds']
INTERVAL_SECONDS_ADJ = settings['interval_seconds_adj']
DRESPONSE_VALUE = settings['dresponse_value']


print(settings)


{'WRF_dir': '/Users/ngordillo/florence/', 'adj_jc': 119, 'adj_ic': 181, 'box_size': 10, 'run_hours': '36', 'start_year': '2018', 'start_month': '09', 'start_day': '09', 'start_hour': '00', 'end_year': '2018', 'end_month': '09', 'end_day': '10', 'end_hour': '12', 'interval_seconds': '3600', 'interval_seconds_adj': '10800', 'time_step': '60', 'e_we': 350, 'e_sn': 240, 'dx': 18000, 'dy': 18000, 'dresponse_value': -150}


In [3]:
os.chdir(WRF_DIR)
command_mkdir = "mkdir exp_files"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/")
command_mkdir = "mkdir " + EXP_NAME
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/" + EXP_NAME)
        

command_mkdir = "mkdir namelists"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

command_mkdir = "mkdir wrf_data"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)


mkdir: cannot create directory ‘exp_files’: File exists
mkdir: cannot create directory ‘WRF_Florence_test’: File exists
mkdir: cannot create directory ‘namelists’: File exists
mkdir: cannot create directory ‘wrf_data’: File exists


In [ ]:
# Generate namelist for trajectory run

output_file = os.path.join(WRF_DIR, "exp_files/" + EXP_NAME + "/namelists/namelist.input.trj." + EXP_NAME) 

proceed = True

if os.path.exists(output_file):
    response = input(f"Warning: A namelist already exists at {output_file}. Overwrite it? (y/n): ")
    
    if response.lower() not in ['y', 'yes']:
        print("Skipping namelist generation.")
        proceed = False 
        

if(proceed):
    make_namelist.generate_namelist(
        run_hours = RUN_HOURS,
        start_year = START_YEAR,
        start_month = START_MONTH,
        start_day = START_DAY,
        start_hour = START_HOUR,
        end_year = END_YEAR,
        end_month = END_MONTH,
        end_day = END_DAY,
        end_hour = END_HOUR,
        time_step = TIME_STEP,
        interval_seconds = INTERVAL_SECONDS,
        e_we = E_WE,
        e_sn = E_SN,
        dx = DX,
        dy = DY,    
        wrf_dir = WRF_DIR,
        exp_name = EXP_NAME,

        run_type="trj"

)


Success! 'namelist.input.trj.WRF_Florence_test' has been generated.


## Section 1: Run Tracjectory

In [6]:

#Link TRJ namelist to namelist.input

os.chdir(WRF_DIR + 'em_adj/')
command_linktrj = "ln -sf exp_files/" + EXP_NAME + "/namelists/namelist.input.trj." + EXP_NAME + " namelist.input"
output_linktrj=subprocess.run(command_linktrj,shell=True, stdout=PIPE)


In [6]:
# Run WRF model with trajectory data assimilation (or do in terminal)
command_runwrf = "source ~/.bashrc && mpirun -np 40 ./wrf.exe"
output_runwrf = subprocess.run(command_runwrf, shell=True, executable='/bin/bash', stdout=subprocess.PIPE)

 starting wrf task            5  of           40
 starting wrf task            7  of           40
 starting wrf task            9  of           40
 starting wrf task           11  of           40
 starting wrf task           16  of           40
 starting wrf task           17  of           40
 starting wrf task           18  of           40
 starting wrf task           19  of           40
 starting wrf task           21  of           40
 starting wrf task           25  of           40
 starting wrf task           31  of           40
 starting wrf task           32  of           40
 starting wrf task           33  of           40
 starting wrf task           34  of           40
 starting wrf task           36  of           40
 starting wrf task           37  of           40
 starting wrf task            0  of           40
 starting wrf task            1  of           40
 starting wrf task            2  of           40
 starting wrf task            3  of           40
 starting wrf task  

In [7]:
# After WRF run, copy output files to a separate directory to save

os.chdir(WRF_DIR + 'em_adj/')
command_mkdir = "mkdir save"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

command_cp = "cp wrfout_d01_* save/"
output_cp=subprocess.run(command_cp,shell=True, stdout=PIPE)


mkdir: cannot create directory ‘save’: File exists
cp: cannot stat 'wrfout_d01_2026-09-10_12:00:00': No such file or directory


## Section 2: Prepare and Run Adjoint

In [5]:
# Copy the final output file for adjoint run

os.chdir(WRF_DIR + 'em_adj/')
command_cp = "cp " + WRF_DIR + "em_adj/wrfout_d01_"+str(END_YEAR)+"-"+str(END_MONTH)+"-"+str(END_DAY)+"_"+str(END_HOUR)+":00:00 " + WRF_DIR + "em_adj/final_sens_d01_"+str(END_YEAR)+"-"+str(END_MONTH)+"-"+str(END_DAY)+"_"+str(END_HOUR)+":00:00"
output_trj=subprocess.run(command_cp,shell=True, stdout=PIPE)



cp: '/Users/ngordillo/florence/em_adj/wrfout_d01_2018-09-10_12:00:00' and '/Users/ngordillo/florence/em_adj/final_sens_d01_2018-09-10_12:00:00' are the same file


In [6]:
# Open the netCDF files for forward and adjoint runs (identical files))

nc_fwd_fname = WRF_DIR + 'em_adj/wrfout_d01_'+str(END_YEAR)+'-'+str(END_MONTH)+'-'+str(END_DAY)+'_'+str(END_HOUR)+':00:00'    # Your filename
wrfout = Dataset(nc_fwd_fname, 'r')  # Dataset is the class behavior to open the file
#
nc_adj_fname = WRF_DIR + 'em_adj/final_sens_d01_'+str(END_YEAR)+'-'+str(END_MONTH)+'-'+str(END_DAY)+'_'+str(END_HOUR)+':00:00'    # Your filename
adjout = Dataset(nc_adj_fname, 'r+')  # Dataset is the class behavior to open the file

In [7]:
#Get variables for forward and adjoint runs (identical files)

itime = -1 # select model time
slp=getvar(wrfout, "slp", timeidx=itime,meta=False)   # Map scale factor on mass grid
msfm=getvar(wrfout, "MAPFAC_M", timeidx=itime,meta=False)   # Map scale factor on mass grid
u=getvar(wrfout, "U", timeidx=itime,meta=False)   # u-wind for grid size
num_levs = u.shape[0]
num_lats = msfm.shape[0]
num_lons = msfm.shape[1]
ds = 18.e3

In [ ]:
#Calculate the gradient of the cost function with respect to the surface pressure (mu) at the surface (isobaric level)

g_mu=np.zeros([num_lats,num_lons])

#
n = 0
jc = ADJ_JC
ic = ADJ_IC 

for j in np.arange(jc-BOX_SIZE,jc+BOX_SIZE):
    for i in np.arange(ic-BOX_SIZE,ic+BOX_SIZE): 
        # if(slp[j,i]<=984):
            g_mu[j,i]= -1.0
            n += 1
print(n)
g_mu = g_mu/n

# Write the gradient of the cost function with respect to mu at the surface (isobaric level) to the adjoint output file

adjout.variables['G_MU'][0,:,:] = g_mu[:,:]
adjout.close()

400


In [ ]:
# Generate namelist for adjoint run

output_file = os.path.join(WRF_DIR, "exp_files/" + EXP_NAME + "/namelists/namelist.input.adj." + EXP_NAME) 

proceed = True

if os.path.exists(output_file):
    response = input(f"Warning: A namelist already exists at {output_file}. Overwrite it? (y/n): ")
    
    if response.lower() not in ['y', 'yes']:
        print("Skipping namelist generation.")
        proceed = False 
        

if(proceed):
    make_namelist.generate_namelist(
        run_hours = RUN_HOURS,
        start_year = START_YEAR,
        start_month = START_MONTH,
        start_day = START_DAY,
        start_hour = START_HOUR,
        end_year = END_YEAR,
        end_month = END_MONTH,
        end_day = END_DAY,
        end_hour = END_HOUR,
        time_step = TIME_STEP,
        interval_seconds = INTERVAL_SECONDS_ADJ,
        e_we = E_WE,
        e_sn = E_SN,
        dx = DX,
        dy = DY,    
        wrf_dir = WRF_DIR,
        exp_name = EXP_NAME,

        run_type="adj"

)


Success! 'namelist.input.adj.WRF_Florence_test' has been generated.


In [11]:
#Link the final output file for adjoint run to the namelist.input file and copy to a separate file 

command_link = "ln -sf " + WRF_DIR + "em_adj/final_sens_d01_"+str(END_YEAR)+"-"+str(END_MONTH)+"-"+str(END_DAY)+"_"+str(END_HOUR)+":00:00 " + WRF_DIR + "em_adj/wrfout_d01_"+str(END_YEAR)+"-"+str(END_MONTH)+"-"+str(END_DAY)+"_"+str(END_HOUR)+":00:00"
output_trj = subprocess.run(command_link,shell=True, stdout=PIPE)

command_cp3 = "cp " + WRF_DIR + "em_adj/final_sens_d01_"+str(END_YEAR)+"-"+str(END_MONTH)+"-"+str(END_DAY)+"_"+str(END_HOUR)+":00:00 " + WRF_DIR + "em_adj/final_sens_d01"
output_cp3 = subprocess.run(command_cp3,shell=True, stdout=PIPE)

command_cp2 = "ln -sf " + WRF_DIR + "exp_files/" + EXP_NAME + "/namelists/namelist.input.adj." + EXP_NAME + " " + WRF_DIR + "em_adj/namelist.input"
print(command_cp2)
output_cp2 = subprocess.run(command_cp2,shell=True, stdout=PIPE)

ln -sf /Users/ngordillo/florence/exp_files/WRF_Florence_test/namelists/namelist.input.adj.WRF_Florence_test /Users/ngordillo/florence/em_adj/namelist.input


In [15]:
#Run WRF adjoint model (or do in terminal)

command_runwrf = "source ~/.bashrc && mpirun -np 40 ./wrf.exe"

output_runwrf = subprocess.run(command_runwrf, shell=True, executable='/bin/bash', stdout=subprocess.PIPE)

 starting wrf task           39  of           40
 starting wrf task            1  of           40
 starting wrf task            6  of           40
 starting wrf task           13  of           40
 starting wrf task           14  of           40
 starting wrf task           16  of           40
 starting wrf task           29  of           40
 starting wrf task           33  of           40
 starting wrf task           36  of           40
 starting wrf task            0  of           40
 starting wrf task            2  of           40
 starting wrf task            3  of           40
 starting wrf task            4  of           40
 starting wrf task            8  of           40
 starting wrf task            9  of           40
 starting wrf task           10  of           40
 starting wrf task           15  of           40
 starting wrf task           18  of           40
 starting wrf task           19  of           40
 starting wrf task           20  of           40
 starting wrf task  

In [ ]:
# After WRF run, copy output files to a separate directory to save

command_cp = "cp " + WRF_DIR + "em_adj/adjout_d01_* " + WRF_DIR + "exp_files/" + EXP_NAME + "/wrf_data/"
output_cp=subprocess.run(command_cp,shell=True, stdout=PIPE)

command_cp = "cp " + WRF_DIR + "em_adj/wrfinput_d01 " + WRF_DIR + "exp_files/" + EXP_NAME + "/wrf_data/"
output_cp=subprocess.run(command_cp,shell=True, stdout=PIPE)

command_cp = "cp " + WRF_DIR + "em_adj/wrfbdy_d01 " + WRF_DIR + "exp_files/" + EXP_NAME + "/wrf_data/"
output_cp=subprocess.run(command_cp,shell=True, stdout=PIPE)


mkdir: cannot create directory ‘save’: File exists
cp: cannot stat '/Users/ngordillo/florence/em_adj/namelist.input.trj': No such file or directory
cp: cannot stat '/Users/ngordillo/florence/em_adj/namelist.input.adj': No such file or directory


## Section 4: Merge 

In [ ]:
# Move to the directory
os.chdir(os.path.join(WRF_DIR, 'em_adj/'))

# Use glob to find files matching your patterns
# This returns a list of strings directly
trajectory_files = sorted(glob.glob("wrfout_d01*"))
adjoint_files = sorted(glob.glob("adjout_d01*"))

# Print the results so you can actually see them
print("Trajectory Files found:")
print(*trajectory_files, sep="\n")

print("\nAdjoint Files found:")
print(*adjoint_files, sep="\n")


num_files=repr(len(adjoint_files))
print(num_files)

Trajectory Files found:
wrfout_d01_2018-09-09_00:00:00
wrfout_d01_2018-09-09_01:00:00
wrfout_d01_2018-09-09_02:00:00
wrfout_d01_2018-09-09_03:00:00
wrfout_d01_2018-09-09_04:00:00
wrfout_d01_2018-09-09_05:00:00
wrfout_d01_2018-09-09_06:00:00
wrfout_d01_2018-09-09_07:00:00
wrfout_d01_2018-09-09_08:00:00
wrfout_d01_2018-09-09_09:00:00
wrfout_d01_2018-09-09_10:00:00
wrfout_d01_2018-09-09_11:00:00
wrfout_d01_2018-09-09_12:00:00
wrfout_d01_2018-09-09_13:00:00
wrfout_d01_2018-09-09_14:00:00
wrfout_d01_2018-09-09_15:00:00
wrfout_d01_2018-09-09_16:00:00
wrfout_d01_2018-09-09_17:00:00
wrfout_d01_2018-09-09_18:00:00
wrfout_d01_2018-09-09_19:00:00
wrfout_d01_2018-09-09_20:00:00
wrfout_d01_2018-09-09_21:00:00
wrfout_d01_2018-09-09_22:00:00
wrfout_d01_2018-09-09_23:00:00
wrfout_d01_2018-09-10_00:00:00
wrfout_d01_2018-09-10_01:00:00
wrfout_d01_2018-09-10_02:00:00
wrfout_d01_2018-09-10_03:00:00
wrfout_d01_2018-09-10_04:00:00
wrfout_d01_2018-09-10_05:00:00
wrfout_d01_2018-09-10_06:00:00
wrfout_d01_2018

In [20]:
# Copy the adjoint variables to the trajectory files for each time step

for i in range(int(num_files)):
    print(adjoint_files[i],trajectory_files[i])
    adj=Dataset(adjoint_files[i],'r')
    wrf=Dataset(trajectory_files[i],'a')

    wrf.variables['A_T'][:]      = adj.variables['A_T'][:]
    wrf.variables['A_U'][:]      = adj.variables['A_U'][:]
    wrf.variables['A_V'][:]      = adj.variables['A_V'][:]
    wrf.variables['A_P'][:]      = adj.variables['A_P'][:]
    wrf.variables['A_MU'][:]     = adj.variables['A_MU'][:]
    wrf.variables['A_QVAPOR'][:] = adj.variables['A_QVAPOR'][:]
    wrf.variables['A_W'][:]      = adj.variables['A_W'][:]

wrf.close()


adjout_d01_2018-09-09_00:00:00 wrfout_d01_2018-09-09_00:00:00
adjout_d01_2018-09-09_01:00:00 wrfout_d01_2018-09-09_01:00:00
adjout_d01_2018-09-09_02:00:00 wrfout_d01_2018-09-09_02:00:00
adjout_d01_2018-09-09_03:00:00 wrfout_d01_2018-09-09_03:00:00
adjout_d01_2018-09-09_04:00:00 wrfout_d01_2018-09-09_04:00:00
adjout_d01_2018-09-09_05:00:00 wrfout_d01_2018-09-09_05:00:00
adjout_d01_2018-09-09_06:00:00 wrfout_d01_2018-09-09_06:00:00
adjout_d01_2018-09-09_07:00:00 wrfout_d01_2018-09-09_07:00:00
adjout_d01_2018-09-09_08:00:00 wrfout_d01_2018-09-09_08:00:00
adjout_d01_2018-09-09_09:00:00 wrfout_d01_2018-09-09_09:00:00
adjout_d01_2018-09-09_10:00:00 wrfout_d01_2018-09-09_10:00:00
adjout_d01_2018-09-09_11:00:00 wrfout_d01_2018-09-09_11:00:00
adjout_d01_2018-09-09_12:00:00 wrfout_d01_2018-09-09_12:00:00
adjout_d01_2018-09-09_13:00:00 wrfout_d01_2018-09-09_13:00:00
adjout_d01_2018-09-09_14:00:00 wrfout_d01_2018-09-09_14:00:00
adjout_d01_2018-09-09_15:00:00 wrfout_d01_2018-09-09_15:00:00
adjout_d

In [ ]:
#Consider merging the trajectory files into a single file for easier analysis if ncrcat is available (optional)

#/usr/local/netcdf4_gfortran/bin/ncrcat -O /Users/morgan/florence/mu/em_adj/wrfout_d01*00 wrfout_florence_mu.nc

### Section 5: Calculate TLM Linearity Ratio 

In [15]:
#Load the adjoint variables from gradient file.

adj_filename = WRF_DIR +'em_adj/gradient_wrfplus_d01_' + START_YEAR + '-' + START_MONTH + '-' + START_DAY + '_' + START_HOUR + ':00:00'    # Your filename
adj_file = Dataset(adj_filename, 'r')  # Dataset is the class behavior to open the file

In [16]:
#Load the final sensitivity file for the forward run to get the grid information

final_sens_file = WRF_DIR + '/em_adj/final_sens_d01'    # Your filename
final_time_file = Dataset(final_sens_file, 'r')  # Dataset is the class behavior to open the file

itime = -1 # select model time
msfm=getvar(final_time_file, "MAPFAC_M", timeidx=itime,meta=False)   # Map scale factor on mass grid
slp=getvar(final_time_file, "slp", timeidx=itime,meta=False)   # Map scale factor on mass grid
mup=getvar(final_time_file, "MU", timeidx=itime,meta=False)   # Map scale factor on mass grid
G_MU=getvar(final_time_file, "G_MU", timeidx=itime,meta=False)   # Map scale factor on mass grid
u = getvar(final_time_file, "U",timeidx=-1,meta=False)
ds = 1./final_time_file.variables['RDX'][0]     # grid spacing SAME IN ZONAL AND MERIDIONAL DIRECTIONS


num_levs=u.shape[0]
num_lats = msfm.shape[0]
num_lons = msfm.shape[1]

In [17]:
#Calculate R_mu, the response of the cost function to a perturbation in mu at the surface (isobaric level)

R_mu=0

#
n = 0
jc = ADJ_JC
ic = ADJ_IC

for j in np.arange(jc-BOX_SIZE,jc+BOX_SIZE):
    for i in np.arange(ic-BOX_SIZE,ic+BOX_SIZE): 
        #if(G_MU[j,i]<=0):
            R_mu += -mup[j,i]
            n += 1
print(n)            
R_mu = R_mu/n
print(n, R_mu)    

400
400 455.2194415449025


In [18]:
#Calculate du, dv, dT, dq, the perturbations to the initial conditions that would lead to a perturbation in mu at the surface (isobaric level) equal to dR

dR=DRESPONSE_VALUE
print(dR)

uin = adj_file.variables['A_U'][0,:] # u-wind
vin = adj_file.variables['A_V'][0,:] # v-wind
tin = adj_file.variables['A_T'][0,:] # T
qin = adj_file.variables['A_QVAPOR'][0,:] # Q

cp   = 1004
Tbar = 300
P0   = 100000
R    = 287.04
L    = 2.5104*10**6
lam  = dR/( np.sum(uin*uin) + np.sum(vin*vin) + (Tbar/cp)*np.sum(tin*tin) + (cp*Tbar/L**2)*np.sum(qin*qin) )
print('lam = ', lam)

du,dv,dT,dq = (lam*uin,lam*vin, lam*Tbar*tin/cp, lam*(cp*Tbar/L**2)*qin )

-150
lam =  -56.85269792832733


In [ ]:
import xarray as xr

# Calculate the maximum absolute values of du, dv, dT, dq for normalization and save to a text file and netCDF file using xarray

print(np.amax(np.abs((du))))
print(np.amax(np.abs((dv))))
print(np.amax(np.abs((dT))))
print(np.amax(np.abs((dq))))

max_du = np.amax(np.abs(du))
max_dv = np.amax(np.abs(dv))
max_dT = np.amax(np.abs(dT))
max_dq = np.amax(np.abs(dq))

indices = np.where(du==du.max())
print(indices)

with open(WRF_DIR + "exp_files/" + EXP_NAME + '/wrf_data/max.txt', 'w') as file:
    file.write(f"{max_du}\n")
    file.write(f"{max_dv}\n")
    file.write(f"{max_dT}\n")
    file.write(f"{max_dq}\n")

print("Values successfully saved to 'amaxs_output.txt'")

# Save du, dv, dT, dq to a NetCDF file using xarray

output_nc_file = WRF_DIR + "exp_files/" + EXP_NAME + '/wrf_data/perturbations.nc'
ds = xr.Dataset()
ds['du'] = (['level', 'lat', 'lon_u'], du)
ds['dv'] = (['level', 'lat_v', 'lon'], dv)
ds['dT'] = (['level', 'lat', 'lon'], dT)
ds['dq'] = (['level', 'lat', 'lon'], dq)
ds.to_netcdf(output_nc_file)

print("Perturbations successfully saved to 'perturbations.nc'")

1.3009295273800956
1.1874580721646815
2.447225425286064
8.200395913584565e-05
(array([14]), array([125]), array([188]))
Values successfully saved to 'amaxs_output.txt'
Perturbations successfully saved to 'perturbations.nc'


### Section 6: Prepare for TLM (and FWD_Opt)

In [4]:
# Copy the starting output file of the trajectory run as the initial conditions to perturb

os.chdir(WRF_DIR + 'em_tlm/')
command_cp = "cp " + WRF_DIR + "em_adj/wrfout_d01_"+ START_YEAR + "-" + START_MONTH + "-" + START_DAY + "_" + START_HOUR + ":00:00 " + WRF_DIR + "em_tlm/init_pert_d01"
output_trj=subprocess.run(command_cp,shell=True, stdout=PIPE)

In [ ]:
# Load the perturbations from the NetCDF file and add them to the initial conditions for the TLM run

perts = xr.open_dataset(WRF_DIR + "exp_files/" + EXP_NAME + '/wrf_data/perturbations.nc')

du = perts['du'].values
dv = perts['dv'].values
dT = perts['dT'].values
dq = perts['dq'].values

perts.close()


nc_fname = WRF_DIR + 'em_tlm/init_pert_d01'  # Your filename
ncf_fwd_tlm_new = Dataset(nc_fname, 'r+')  # Dataset is the class behavior to open the file

#
ncf_fwd_tlm_new.variables['G_U'][:] = ncf_fwd_tlm_new.variables['G_U'][:] + du
ncf_fwd_tlm_new.variables['G_V'][:] = ncf_fwd_tlm_new.variables['G_V'][:] + dv
ncf_fwd_tlm_new.variables['G_T'][:] = ncf_fwd_tlm_new.variables['G_T'][:] + dT
ncf_fwd_tlm_new.variables['G_QVAPOR'][:] = ncf_fwd_tlm_new.variables['G_QVAPOR'][:] + dq    
#
ncf_fwd_tlm_new.close()


In [ ]:
# Copy wrfinput_d01 to em_fwd_opt 

os.chdir(WRF_DIR + 'em_fwd_opt/')
command_cp = "cp /Users/brookezibton/s4_folders/s4_data/WRF_DATA/FLORENCE_201809090000/NCEP_18km/wrf_folder/wrfinput_d01 ."
#
output_trj=subprocess.run(command_cp,shell=True, stdout=PIPE)


In [ ]:


new_IC_filename = WRF_DIR + 'em_fwd_opt/wrfinput_d01'  # Your filename
perturbed_IC = Dataset(new_IC_filename, 'r+')  # Dataset is the class behavior to open the file

u0 = perturbed_IC.variables['U']
v0 = perturbed_IC.variables['V']
T0 = perturbed_IC.variables['T']
q0 = perturbed_IC.variables['QVAPOR']

print(u0[0,3,115,170],du[3,115,170])

utotal = np.asarray(u0) + np.asarray(du)
vtotal = np.asarray(v0) + np.asarray(dv)
Ttotal = np.asarray(T0) + np.asarray(dT)
Qtotal = np.asarray(q0) + np.asarray(dq)

u0[:,:,:,:] = utotal
v0[:,:,:,:] = vtotal
T0[:,:,:,:] = Ttotal
q0[:,:,:,:] = Qtotal

print(u0[0,3,115,170])
perturbed_IC.close()

In [ ]:
new_IC_filename = WRF_DIR + 'em_fwd_opt/wrfinput_d01'  # Your filename
perturbed_IC = Dataset(new_IC_filename, 'r+')  # Dataset is the class behavior to open the file
u0 = perturbed_IC.variables['U']
print(u0[0,3,115,170])

print(dR)

dR_check=np.sum(uin*du) + np.sum(vin*dv) + np.sum(tin*dT)+ np.sum(qin*dq)
print('Expected change: ',dR_check, 'compared with specified change: ', dR)

### Section 7: Run TLM

In [ ]:
# make dirs for namelist, and outputsif they don't exist and wrfinputs if they don't exist

output_file = os.path.join(WRF_DIR, "exp_files/" + EXP_NAME + "/namelists/namelist.input.tlm." + EXP_NAME) 

proceed = True

#Comment out the following block if you want to overwrite existing namelist without warning.

if os.path.exists(output_file):
    response = input(f"Warning: A namelist already exists at {output_file}. Overwrite it? (y/n): ")
    
    if response.lower() not in ['y', 'yes']:
        print("Skipping namelist generation.")
        proceed = False 
        
####

if(proceed):

    make_namelist.generate_namelist(
        run_hours = RUN_HOURS,
        start_year = START_YEAR,
        start_month = START_MONTH,
        start_day = START_DAY,
        start_hour = START_HOUR,
        end_year = END_YEAR,
        end_month = END_MONTH,
        end_day = END_DAY,
        end_hour = END_HOUR,
        time_step = TIME_STEP,
        interval_seconds = INTERVAL_SECONDS,
        e_we = E_WE,
        e_sn = E_SN,
        dx = DX,
        dy = DY,    
        wrf_dir = WRF_DIR,
        exp_name = EXP_NAME,

        run_type="tlm"

    )

In [ ]:

#Link TLM namelist to namelist.input

os.chdir(WRF_DIR + 'em_tlm/')
command_linktlm = "ln -sf " + WRF_DIR + "exp_files/" + EXP_NAME + "/namelists/namelist.input.tlm." + EXP_NAME + " namelist.input"
# command_linktrj = "ln -sf namelist.input.tlm namelist.input"
output_linktlm=subprocess.run(command_linktlm,shell=True, stdout=PIPE)

command_linkaux = "ln -sf ../em_adj/auxhist6_d01* ."
output_linkaux=subprocess.run(command_linkaux,shell=True, stdout=PIPE)


In [ ]:
#Run TLM 

command_runwrf = "source ~/.bashrc && mpirun -np 40 ./wrf.exe"
output_runwrf = subprocess.run(command_runwrf, shell=True, executable='/bin/bash', stdout=subprocess.PIPE)

### Section 8: Calculate Ratio

In [ ]:
#Load in gradient from adjoint run to compare with the response from the TLM run

nc_fname = WRF_DIR + "em_adj/gradient_wrfplus_d01_"+ START_YEAR + "-" + START_MONTH + "-" + START_DAY + "_"+ START_HOUR + ":00:00"    # Your filename
ncf_adj = Dataset(nc_fname, 'r')  # Dataset is the class behavior to open the file

In [ ]:
#Load in the TLM output file to compare with the adjoint gradient

tlm_file = WRF_DIR + "em_tlm/tlmout_d01_" + END_YEAR + "-" + END_MONTH + "-" + END_DAY + "_"+ END_HOUR + ":00:00"    # Your filename
tlmout = Dataset(tlm_file, 'r')  # Dataset is the class behavior to open the file

trj_file = WRF_DIR + "em_adj/wrfout_d01_" + END_YEAR + "-" + END_MONTH + "-" + END_DAY + "_"+ END_HOUR + ":00:00"    # Your filename
wrfout = Dataset(trj_file, 'r')  # Dataset is the class behavior to open the file

itime = -1 # select model time
msfm=getvar(wrfout, "MAPFAC_M", timeidx=itime,meta=False)   # Map scale factor on mass grid
slp=getvar(wrfout, "slp", timeidx=itime,meta=False)   # Map scale factor on mass grid
g_u = getvar(tlmout, "G_U",timeidx=-1,meta=False)
ds = 30.e3

num_levs=g_u.shape[0]
num_lats = msfm.shape[0]
num_lons = msfm.shape[1]

vorticity = np.zeros([num_levs,num_lats,num_lons])
cor = np.zeros([num_lats,num_lons])

mup = getvar(wrfout, "MU",timeidx=-1,meta=False)
mupp=getvar(tlmout, "G_MU", timeidx=itime,meta=False)   # Map scale factor on mass grid


In [ ]:
#Calculate the response of the cost function to a perturbation in mu at the surface (isobaric level) from the TLM run and compare with the adjoint gradient and specified change in cost function value

R_mu_tlm = 0.0
R_mu = 0.
#
n = 0
jc = ADJ_JC
ic = ADJ_IC

for j in np.arange(jc-BOX_SIZE,jc+BOX_SIZE):
    for i in np.arange(ic-BOX_SIZE,ic+BOX_SIZE): 
        #if(slp[j,i]<=972):
            R_mu_tlm -= mupp[j,i]
            R_mu -= mup[j,i]
            n += 1
            
R_mu_tlm = R_mu_tlm/n
R_mu = R_mu/n
print('TLM change in R_mu = ', R_mu_tlm, 'compared with R_mu = ', DRESPONSE_VALUE, ' with a ratio of: ', (R_mu_tlm)/(DRESPONSE_VALUE))
print('change in R_mu = ', R_mu_tlm, 'ratio of: ', (R_mu_tlm)/(DRESPONSE_VALUE))             